# Bài học 16: Database — Lưu trữ dữ liệu

## Tại sao cần database?

Ở các bài trước, khi agent tạo xong một bài viết, kết quả chỉ hiện lên màn hình rồi **biến mất**. Tắt chương trình là mất sạch.

Trong thực tế, chúng ta cần:
- **Lưu trữ** các bài viết đã tạo
- **Theo dõi** trạng thái (đang xử lý, hoàn thành, lỗi...)
- **Tìm kiếm** và **lọc** bài viết theo tiêu chí
- **Xem lại** lịch sử chỉnh sửa

**Database** giải quyết tất cả những điều này.

## SQLite — Database đơn giản nhất

SQLite được **tích hợp sẵn** trong Python — không cần cài đặt gì thêm. Dữ liệu được lưu vào **một file duy nhất** (ví dụ: `workspace.db`). Rất phù hợp cho ứng dụng vừa và nhỏ.

So sánh:
- **Không có database**: Dữ liệu mất khi tắt chương trình
- **Có SQLite**: Dữ liệu lưu vào file, truy cập lại bất cứ lúc nào

In [ ]:
import sqlite3

# Tạo database trong bộ nhớ (sẽ biến mất khi chạy xong)
# Dùng để thử nghiệm — không ảnh hưởng file thật
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

# Tạo bảng — giống như tạo một bảng tính
conn.execute("""
    CREATE TABLE articles (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        topic TEXT NOT NULL,
        status TEXT DEFAULT 'queued'
    )
""")

# Chèn dữ liệu
conn.execute("INSERT INTO articles (topic) VALUES (?)", ("SEO Guide 2026",))
conn.execute("INSERT INTO articles (topic, status) VALUES (?, ?)", ("Content Marketing", "review"))
conn.commit()

# Đọc dữ liệu
rows = conn.execute("SELECT * FROM articles").fetchall()
for row in rows:
    print(f"#{row['id']}: {row['topic']} ({row['status']})")
conn.close()

## SQL cơ bản

SQL (Structured Query Language) là ngôn ngữ để làm việc với database. Bạn chỉ cần biết 4 lệnh chính:

| Lệnh | Mục đích | Ví dụ |
|------|----------|-------|
| `CREATE TABLE` | Tạo bảng mới | `CREATE TABLE articles (id INTEGER, topic TEXT)` |
| `INSERT INTO` | Thêm dữ liệu | `INSERT INTO articles (topic) VALUES ('SEO Guide')` |
| `SELECT` | Đọc/tìm dữ liệu | `SELECT * FROM articles WHERE status = 'review'` |
| `UPDATE` | Cập nhật dữ liệu | `UPDATE articles SET status = 'done' WHERE id = 1` |

### Kiểu dữ liệu của cột:
- `INTEGER PRIMARY KEY AUTOINCREMENT` — Số ID tự động tăng
- `TEXT NOT NULL` — Chuỗi văn bản bắt buộc
- `TEXT DEFAULT 'queued'` — Văn bản có giá trị mặc định

### Dấu `?` là gì?
Khi truyền dữ liệu vào SQL, dùng `?` thay vì nối chuỗi trực tiếp. Đây là cách **an toàn** để tránh lỗi và các vấn đề bảo mật:
```python
# Đúng:
conn.execute("SELECT * FROM articles WHERE status = ?", ("review",))

# SAI — tuyệt đối không làm thế này:
conn.execute(f"SELECT * FROM articles WHERE status = '{status}'")
```

In [ ]:
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

conn.execute("CREATE TABLE articles (id INTEGER PRIMARY KEY AUTOINCREMENT, topic TEXT, status TEXT DEFAULT 'queued', word_count INTEGER)")
conn.execute("INSERT INTO articles (topic, status, word_count) VALUES (?, ?, ?)", ("SEO Guide", "review", 2000))
conn.execute("INSERT INTO articles (topic, status, word_count) VALUES (?, ?, ?)", ("Content Tips", "queued", None))
conn.execute("INSERT INTO articles (topic, status, word_count) VALUES (?, ?, ?)", ("Link Building", "error", None))
conn.commit()

# Lọc theo trạng thái (WHERE)
print("Các bài đang review:")
for row in conn.execute("SELECT * FROM articles WHERE status = ?", ("review",)).fetchall():
    print(f"  #{row['id']}: {row['topic']} ({row['word_count']} từ)")

# Cập nhật (UPDATE)
conn.execute("UPDATE articles SET status = ?, word_count = ? WHERE id = ?", ("review", 1500, 2))
conn.commit()

print("\nTất cả bài viết sau khi cập nhật:")
for row in conn.execute("SELECT * FROM articles ORDER BY id").fetchall():
    print(f"  #{row['id']}: {row['topic']} - {row['status']} ({row['word_count'] or '-'} từ)")
conn.close()

## Database thật của sản phẩm

Các ví dụ trên là SQL cơ bản. Bây giờ hãy xem **database thật** trong dự án của chúng ta hoạt động như thế nào.

File `db.py` cung cấp các hàm sau:
- `init_db()` — Tạo bảng nếu chưa tồn tại
- `create_article(topic, keywords)` — Tạo bài viết mới (status = 'queued')
- `get_article(id)` — Xem chi tiết một bài viết
- `list_articles()` — Xem tất cả bài viết
- `update_article_status(id, status, **fields)` — Cập nhật trạng thái và nội dung

Pipeline gọi các hàm này để theo dõi tiến trình:
```
queued -> researching -> outlining -> writing -> enriching -> review
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

from dotenv import load_dotenv
load_dotenv()

from db import init_db, create_article, get_article, list_articles, update_article_status

init_db()

# Tạo bài viết thử nghiệm
article_id = create_article("Test Article from Notebook", target_keywords=["test", "notebook"])
print(f"Đã tạo bài viết #{article_id}")

# Đọc lại bài viết
article = get_article(article_id)
print(f"\nChi tiết bài viết:")
print(f"  Chủ đề: {article['topic']}")
print(f"  Trạng thái: {article['status']}")
print(f"  Ngày tạo: {article['created_at']}")

# Cập nhật trạng thái
update_article_status(article_id, "review", word_count=1000, article_markdown="# Test\n\nThis is a test.")
article = get_article(article_id)
print(f"\nSau khi cập nhật:")
print(f"  Trạng thái: {article['status']}")
print(f"  Số từ: {article['word_count']}")

In [ ]:
# Xem tất cả bài viết trong database
articles = list_articles()
print(f"Tổng số bài viết: {len(articles)}\n")
for a in articles:
    print(f"  #{a['id']}: {a['topic']} ({a['status']})")

## Tổng kết

- **Database** lưu trữ dữ liệu lâu dài — dữ liệu vẫn còn khi khởi động lại chương trình
- **SQLite** được tích hợp sẵn trong Python, lưu vào một file duy nhất
- **4 lệnh SQL cơ bản**: `CREATE TABLE`, `INSERT`, `SELECT`, `UPDATE`
- **`db.py`** là tầng database của sản phẩm — cung cấp các hàm để tạo, đọc và cập nhật bài viết
- **Theo dõi trạng thái**: database theo dõi từng bài viết qua các bước pipeline (`queued` -> `researching` -> `writing` -> `enriching` -> `review`)
- Nhờ có theo dõi này, bạn luôn biết bài viết đang ở bước nào trong quy trình

**Bài tiếp theo:** Chúng ta sẽ xây dựng **giao diện dòng lệnh (CLI)** — để bạn có thể chạy pipeline từ terminal mà không cần viết code!

## Bài tập

Sử dụng mô hình database trong bộ nhớ từ phần đầu bài học, hãy viết code thực hiện:

1. Tạo bảng `keywords` với các cột: `id` (integer, tự động tăng), `keyword` (text), `search_volume` (integer)
2. Chèn 3 từ khóa tùy chọn với lượng tìm kiếm tự đặt
3. Truy vấn tất cả từ khóa có lượng tìm kiếm trên 1000
4. In kết quả ra màn hình

Đây chính là mô hình mà `db.py` sử dụng — bạn đang luyện tập nền tảng cơ bản.

In [ ]:
# Bài tập: Viết code của bạn ở đây
